# Notebook 7: RAG Pipeline

## Project: Enterprise Document Intelligence Assistant using LLM and RAG

This is the seventh notebook of the project.

In the previous notebook, we built a semantic retrieval system using FAISS.

In this notebook, we will connect the retriever with a Large Language Model.

This creates the full RAG pipeline.

RAG stands for Retrieval-Augmented Generation.

The system will:

1. Accept a user question.
2. Retrieve the most relevant BBC article chunks using FAISS.
3. Pass the retrieved chunks as context to an LLM.
4. Generate a grounded answer based only on the retrieved context.
5. Display the answer along with the retrieved sources.

The output of this notebook is a working RAG question-answering system.

In [4]:
# Install required libraries
!pip install faiss-cpu sentence-transformers transformers accelerate -q

# Import required libraries
import os
import faiss
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# Find input files automatically
def find_input_file(file_name):
    """
    Searches for a file in common Kaggle locations.
    
    This is useful because files from previous notebooks are usually
    uploaded as input files in Kaggle.
    """
    possible_paths = [
        file_name,
        f"/kaggle/working/{file_name}"
    ]

    # Search inside Kaggle input folders
    for root, dirs, files in os.walk("/kaggle/input"):
        if file_name in files:
            possible_paths.append(os.path.join(root, file_name))

    for path in possible_paths:
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        f"{file_name} not found. Please upload it as input to this notebook."
    )


# Load retriever files
def load_retriever(index_file, metadata_file):
    """
    Loads FAISS index and chunk metadata.
    """
    index_path = find_input_file(index_file)
    metadata_path = find_input_file(metadata_file)

    index = faiss.read_index(index_path)
    metadata = pd.read_csv(metadata_path)

    print("Retriever files loaded successfully.")
    print("FAISS index:", index_path)
    print("Metadata:", metadata_path)
    print("Number of vectors:", index.ntotal)
    print("Metadata shape:", metadata.shape)

    if index.ntotal != len(metadata):
        raise ValueError("FAISS index and metadata size do not match.")

    return index, metadata


# Load embedding model
def load_embedding_model(model_name):
    """
    Loads the same embedding model used in Notebook 4.
    """
    model = SentenceTransformer(model_name)

    print("Embedding model loaded successfully.")
    print("Embedding model:", model_name)

    return model


#  Load LLM
def load_llm(model_name):
    """
    Loads the text generation model and tokenizer for answer generation.
    This version does not use Hugging Face pipeline, so it avoids pipeline task errors.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    print("LLM loaded successfully.")
    print("LLM model:", model_name)
    print("Device:", device)

    return tokenizer, model, device


# Retrieve relevant chunks
def retrieve_chunks(query, embedding_model, index, metadata, top_k=5):
    """
    Retrieves the top-k most relevant chunks for a user question.
    """
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        row = metadata.iloc[idx]

        results.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "chunk_id": row["chunk_id"],
            "doc_id": row["doc_id"],
            "title": row["title"],
            "category": row["category"],
            "chunk_text": row["chunk_text"]
        })

    return pd.DataFrame(results)


# Build RAG prompt
def build_prompt(question, retrieved_chunks):
    """
    Builds a prompt using the retrieved context and user question.
    """
    context = ""

    for _, row in retrieved_chunks.iterrows():
        context += f"Source {row['rank']}:\n"
        context += f"Title: {row['title']}\n"
        context += f"Category: {row['category']}\n"
        context += f"Text: {row['chunk_text']}\n\n"

    prompt = f"""
You are an enterprise document intelligence assistant.

Answer the question using only the context provided below.

If the answer is not available in the context, say:
"I do not have enough information in the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt


# Generate answer
def generate_answer(prompt, tokenizer, model, device, max_new_tokens=200):
    """
    Generates an answer using model.generate().
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer.strip()


# Full RAG pipeline
def ask_rag(question, embedding_model, index, metadata, tokenizer, llm_model, device, top_k=5):
    """
    Complete RAG pipeline:
    question -> retrieve chunks -> build prompt -> generate answer
    """
    retrieved_chunks = retrieve_chunks(
        query=question,
        embedding_model=embedding_model,
        index=index,
        metadata=metadata,
        top_k=top_k
    )

    prompt = build_prompt(question, retrieved_chunks)

    answer = generate_answer(
        prompt=prompt,
        tokenizer=tokenizer,
        model=llm_model,
        device=device
    )

    return answer, retrieved_chunks

# Display final RAG output
def display_rag_output(question, answer, retrieved_chunks):
    """
    Displays generated answer and retrieved sources.
    """
    print("=" * 100)
    print("Question:")
    print(question)

    print("\nGenerated Answer:")
    print(answer)

    print("\nRetrieved Sources:")
    display(retrieved_chunks[[
        "rank",
        "score",
        "chunk_id",
        "doc_id",
        "title",
        "category"
    ]])

    print("\nRetrieved Context Preview:")

    for _, row in retrieved_chunks.iterrows():
        print("=" * 100)
        print("Rank:", row["rank"])
        print("Score:", round(row["score"], 4))
        print("Title:", row["title"])
        print("Category:", row["category"])
        print("\nChunk Preview:\n")
        print(row["chunk_text"][:700])
        print()





In [5]:
# Load FAISS index and metadata
index_file = "bbc_faiss_index.index"
metadata_file = "bbc_chunk_metadata.csv"

faiss_index, metadata = load_retriever(index_file, metadata_file)


# Load embedding model
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)


# Load LLM
LLM_MODEL_NAME = "google/flan-t5-base"

llm_tokenizer, llm_model, device = load_llm(LLM_MODEL_NAME)


# Ask a sample question
question = "What are the main updates related to the economy and financial markets?"

answer, retrieved_chunks = ask_rag(
    question=question,
    embedding_model=embedding_model,
    index=faiss_index,
    metadata=metadata,
    tokenizer=llm_tokenizer,
    llm_model=llm_model,
    device=device,
    top_k=5
)

display_rag_output(question, answer, retrieved_chunks)


# Try multiple questions
sample_questions = [
    "What are the main updates related to politics?",
    "What is happening in sports news?",
    "What are the key business and economy updates?",
    "What technology-related news is discussed?",
    "What entertainment news is available?"
]

rag_results = []

for question in sample_questions:
    answer, retrieved_chunks = ask_rag(
        question=question,
        embedding_model=embedding_model,
        index=faiss_index,
        metadata=metadata,
        tokenizer=llm_tokenizer,
        llm_model=llm_model,
        device=device,
        top_k=5
    )

    rag_results.append({
        "question": question,
        "answer": answer,
        "top_source_title": retrieved_chunks.iloc[0]["title"],
        "top_source_category": retrieved_chunks.iloc[0]["category"],
        "top_similarity_score": retrieved_chunks.iloc[0]["score"]
    })


# Save sample RAG results
rag_results_df = pd.DataFrame(rag_results)

print("\nSample RAG Results:")
display(rag_results_df)

output_file = "sample_rag_results.csv"

rag_results_df.to_csv(output_file, index=False)

print("\nSample RAG results saved successfully.")
print("Output file:", output_file)

Retriever files loaded successfully.
FAISS index: /kaggle/input/datasets/jahnavidulala/faiss-index/bbc_faiss_index.index
Metadata: /kaggle/input/datasets/jahnavidulala/chunk-metadata/bbc_chunk_metadata.csv
Number of vectors: 8622
Metadata shape: (8622, 7)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully.
Embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

...

LLM loaded successfully.
LLM model: google/flan-t5-base
Device: cpu
Question:
What are the main updates related to the economy and financial markets?

Generated Answer:
Sunak's 2022 Spring Statement

Retrieved Sources:


,rank,score,chunk_id,doc_id,title,category
0,1,0.472694,158_1,158,BBC editors react to Sunak's 2022 Spring State...,unknown
1,2,0.430077,2484_1,2484,Bank of England set to raise interest rates again,unknown
2,3,0.424829,7484_1,7484,What taxes might Rachel Reeves raise?,unknown
3,4,0.424008,1194_1,1194,IMF: UK set for slowest growth of G7 countries...,unknown
4,5,0.423652,8349_1,8349,UK borrowing costs at highest for a year after...,unknown



Retrieved Context Preview:
Rank: 1
Score: 0.4727
Title: BBC editors react to Sunak's 2022 Spring Statement
Category: unknown

Chunk Preview:

Laura Kuenssberg, Faisal Islam and Simon Jack review the headlines and reaction to the chancellor's update on the UK economy.

Rank: 2
Score: 0.4301
Title: Bank of England set to raise interest rates again
Category: unknown

Chunk Preview:

Interest rates are expected to rise again as the cost if living goes up, so how could this affect you?

Rank: 3
Score: 0.4248
Title: What taxes might Rachel Reeves raise?
Category: unknown

Chunk Preview:

The chancellor says taxes will have to rise to fix the public finances - but which ones could go up?

Rank: 4
Score: 0.424
Title: IMF: UK set for slowest growth of G7 countries in 2023
Category: unknown

Chunk Preview:

The IMF cuts its UK forecast for 2023 and says the global economy is "teetering on the edge" of recession.

Rank: 5
Score: 0.4237
Title: UK borrowing costs at highest for a year after Budget

,question,answer,top_source_title,top_source_category,top_similarity_score
0,What are the main updates related to politics?,I do not have enough information in the provid...,Has frantic election campaign actually grapple...,unknown,0.483725
1,What is happening in sports news?,World Cup 2022: What we learned from a thrilli...,Key rivalries to watch out for at Paris 2024,unknown,0.517542
2,What are the key business and economy updates?,I do not have enough information in the provid...,BBC editors react to Sunak's 2022 Spring State...,unknown,0.472559
3,What technology-related news is discussed?,I do not have enough information in the provid...,Ros Atkins: My list of things I don't understa...,unknown,0.467551
4,What entertainment news is available?,I do not have enough information in the provid...,Ros Atkins: My list of things I don't understa...,unknown,0.466247



Sample RAG results saved successfully.
Output file: sample_rag_results.csv
